# Handwritten Formula Recognition - Model Inference Notebook (`predict1.ipynb`)

This notebook performs inference on handwritten mathematical formula images using the trained **FormulaRecognizer** (EfficientNet-B0 Encoder + Transformer Decoder) model created in `train1.ipynb`.

### Key Highlights:
1. **Environment Setup & Device Selection**: Automated GPU/CPU verification.
2. **Vocabulary Loading**: Loads token mappings from `weights/vocab.pkl` (or trained checkpoint).
3. **Model Architecture**: Replicating `Encoder`, `PositionalEncoding`, `Decoder`, and `FormulaRecognizer` from `train1.ipynb`.
4. **Checkpoint Restoration**: Loads model weights from `checkpoints/best_model.pth`.
5. **Image Preprocessing**: Aspect-ratio preserving `resize_with_padding` (128x512) and ImageNet normalization.
6. **Decoding Strategies**: Autoregressive **Greedy Search** and **Beam Search** with length penalty.
7. **Visualization**: Side-by-side display of handwritten formula images with rendered LaTeX predictions and raw string outputs.

In [ ]:
# ==========================================================
# Cell 1 - Imports
# ==========================================================

import os
import math
import json
import pickle
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

warnings.filterwarnings("ignore")

In [ ]:
# ==========================================================
# Cell 2 - Configuration & Device Setup
# ==========================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

print("=" * 60)
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())
print("Inference Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU Device Name :", torch.cuda.get_device_name(0))
    print("Total VRAM      :", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
print("=" * 60)

In [ ]:
# ==========================================================
# Cell 3 - Hyperparameters (Matching train1.ipynb)
# ==========================================================

IMAGE_HEIGHT = 128
IMAGE_WIDTH = 512
EMBED_DIM = 256
NUM_HEADS = 8
NUM_LAYERS = 4
DROPOUT = 0.1
MAX_FORMULA_LENGTH = 128

In [ ]:
# ==========================================================
# Cell 4 - Vocabulary Definition & Loader
# ==========================================================

from collections import Counter

class Vocabulary:
    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.special_tokens = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
        self.token2idx = {}
        self.idx2token = {}

    def build(self, formulas):
        counter = Counter()
        for formula in formulas:
            tokens = formula.split()
            counter.update(tokens)
        vocab_tokens = [token for token, freq in counter.items() if freq >= self.min_freq]
        all_tokens = self.special_tokens + sorted(vocab_tokens)
        self.token2idx = {token: idx for idx, token in enumerate(all_tokens)}
        self.idx2token = {idx: token for token, idx in self.token2idx.items()}

    def encode(self, formula):
        tokens = formula.split()
        encoded = [self.token2idx["<SOS>"]]
        for token in tokens:
            encoded.append(self.token2idx.get(token, self.token2idx["<UNK>"]))
        encoded.append(self.token2idx["<EOS>"])
        return encoded

    def decode(self, indices):
        tokens = []
        for idx in indices:
            token = self.idx2token.get(idx, "<UNK>")
            if token == "<EOS>":
                break
            if token not in ["<SOS>", "<PAD>"]:
                tokens.append(token)
        return " ".join(tokens)

    def __len__(self):
        return len(self.token2idx)

# Load pre-saved vocabulary
VOCAB_PATH = "weights/vocab.pkl"
if not os.path.exists(VOCAB_PATH):
    VOCAB_PATH = "vocab.pkl"

with open(VOCAB_PATH, "rb") as f:
    vocab = pickle.load(f)

PAD_IDX = vocab.token2idx["<PAD>"]
SOS_IDX = vocab.token2idx["<SOS>"]
EOS_IDX = vocab.token2idx["<EOS>"]
UNK_IDX = vocab.token2idx["<UNK>"]

print("=" * 60)
print("✅ Vocabulary Loaded Successfully!")
print(f"Vocabulary Size : {len(vocab)}")
print(f"PAD Index       : {PAD_IDX}")
print(f"SOS Index       : {SOS_IDX}")
print(f"EOS Index       : {EOS_IDX}")
print("=" * 60)

In [ ]:
# ==========================================================
# Cell 5 - Model Architecture (Encoder & Transformer Decoder)
# ==========================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((4, 16))
        self.projection = nn.Conv2d(1280, EMBED_DIM, kernel_size=1)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.projection(x)
        B, C, H, W = x.shape
        x = x.flatten(2)       # (B, C, 64)
        x = x.transpose(1, 2)  # (B, 64, C)
        return x


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), EMBED_DIM, padding_idx=PAD_IDX)
        self.position = PositionalEncoding(EMBED_DIM, DROPOUT)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=EMBED_DIM,
            nhead=NUM_HEADS,
            dim_feedforward=1024,
            dropout=DROPOUT,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=NUM_LAYERS)
        self.fc = nn.Linear(EMBED_DIM, len(vocab))

    def forward(self, memory, tgt, tgt_mask=None, tgt_padding_mask=None):
        tgt = self.embedding(tgt)
        tgt = self.position(tgt)
        out = self.decoder(tgt=tgt, memory=memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_padding_mask)
        out = self.fc(out)
        return out


class FormulaRecognizer(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, images, tgt, tgt_mask=None, tgt_padding_mask=None):
        memory = self.encoder(images)
        return self.decoder(memory, tgt, tgt_mask, tgt_padding_mask)

In [ ]:
# ==========================================================
# Cell 6 - Load Best Trained Model Weights
# ==========================================================

CHECKPOINT_PATH = "checkpoints/best_model.pth"
model = FormulaRecognizer().to(DEVICE)

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print("=" * 60)
    print("✅ Best Model Loaded Successfully!")
    print(f"Epoch Trained  : {checkpoint.get('epoch', 'N/A') + 1 if isinstance(checkpoint.get('epoch'), int) else checkpoint.get('epoch', 'N/A')}")
    print(f"Best Val Loss  : {checkpoint.get('best_val_loss', 'N/A')}")
    print(f"Model Device   : {next(model.parameters()).device}")
    print("=" * 60)
else:
    print(f"⚠️ Warning: Checkpoint not found at '{CHECKPOINT_PATH}'!")

In [ ]:
# ==========================================================
# Cell 7 - Image Preprocessing Utilities
# ==========================================================

def resize_with_padding(image, target_size=(512, 128)):
    target_w, target_h = target_size
    w, h = image.size
    scale = min(target_w / w, target_h / h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    image = image.resize((new_w, new_h), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (target_w, target_h), (255, 255, 255))
    left = (target_w - new_w) // 2
    top = (target_h - new_h) // 2
    canvas.paste(image, (left, top))
    return canvas

predict_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    padded_image = resize_with_padding(image, (IMAGE_WIDTH, IMAGE_HEIGHT))
    tensor_image = predict_transform(padded_image).unsqueeze(0).to(DEVICE)
    return padded_image, tensor_image

In [ ]:
# ==========================================================
# Cell 8 - Decoding Logic (Greedy Search & Beam Search)
# ==========================================================

def create_causal_mask(seq_len):
    mask = torch.triu(torch.ones(seq_len, seq_len, device=DEVICE), diagonal=1)
    mask = mask.masked_fill(mask == 1, float("-inf"))
    return mask

@torch.no_grad()
def greedy_decode(model, image_tensor, max_length=MAX_FORMULA_LENGTH):
    model.eval()
    memory = model.encoder(image_tensor)
    seq = torch.tensor([[SOS_IDX]], device=DEVICE)
    for _ in range(max_length):
        tgt_mask = create_causal_mask(seq.size(1))
        tgt_padding_mask = (seq == PAD_IDX)
        output = model.decoder(memory=memory, tgt=seq, tgt_mask=tgt_mask, tgt_padding_mask=tgt_padding_mask)
        next_token = torch.argmax(output[:, -1, :], dim=-1).unsqueeze(1)
        seq = torch.cat([seq, next_token], dim=1)
        if next_token.item() == EOS_IDX:
            break
    indices = seq.squeeze().tolist()
    return vocab.decode(indices)

@torch.no_grad()
def beam_search_decode(model, image_tensor, beam_size=5, max_length=MAX_FORMULA_LENGTH, length_penalty=0.7):
    model.eval()
    memory = model.encoder(image_tensor)
    beams = [(torch.tensor([[SOS_IDX]], device=DEVICE), 0.0)]
    completed = []

    for _ in range(max_length):
        candidates = []
        for seq, score in beams:
            if seq[0, -1].item() == EOS_IDX:
                completed.append((seq, score))
                continue
            tgt_mask = create_causal_mask(seq.size(1))
            tgt_padding_mask = (seq == PAD_IDX)
            output = model.decoder(memory=memory, tgt=seq, tgt_mask=tgt_mask, tgt_padding_mask=tgt_padding_mask)
            logits = output[:, -1, :]
            log_probs = F.log_softmax(logits, dim=-1)
            values, indices = torch.topk(log_probs, beam_size, dim=-1)

            for k in range(beam_size):
                token = indices[0, k].view(1, 1)
                new_seq = torch.cat([seq, token], dim=1)
                new_score = score + values[0, k].item()
                candidates.append((new_seq, new_score))

        if not candidates:
            break

        candidates = sorted(candidates, key=lambda x: x[1] / ((x[0].size(1)) ** length_penalty), reverse=True)
        beams = candidates[:beam_size]

    if completed:
        best_seq = max(completed, key=lambda x: x[1] / ((x[0].size(1)) ** length_penalty))[0]
    else:
        best_seq = beams[0][0]

    indices = best_seq.squeeze().tolist()
    return vocab.decode(indices)

In [ ]:
# ==========================================================
# Cell 9 - Integrated Prediction Pipeline
# ==========================================================

def predict_formula(image_path, decode_method="beam", beam_size=5):
    orig_img, tensor_img = preprocess_image(image_path)
    if decode_method == "beam":
        pred_text = beam_search_decode(model, tensor_img, beam_size=beam_size)
    else:
        pred_text = greedy_decode(model, tensor_img)
    return orig_img, pred_text

In [ ]:
# ==========================================================
# Cell 10 - Visualization Function
# ==========================================================

def visualize_prediction(image_path, ground_truth=None, beam_size=5):
    orig_img, greedy_pred = predict_formula(image_path, decode_method="greedy")
    _, beam_pred = predict_formula(image_path, decode_method="beam", beam_size=beam_size)

    fig, axes = plt.subplots(3, 1, figsize=(12, 6), gridspec_kw={'height_ratios': [2, 1, 1]})

    # Input image
    axes[0].imshow(orig_img)
    axes[0].axis("off")
    axes[0].set_title(f"Input Image: {Path(image_path).name}", fontsize=12, fontweight='bold')

    # Greedy prediction rendering
    axes[1].axis("off")
    axes[1].text(0.01, 0.5, f"Greedy: ${greedy_pred}$", fontsize=16, color="navy")
    axes[1].set_title("Greedy Search Output", fontsize=10, loc='left', color='gray')

    # Beam prediction rendering
    axes[2].axis("off")
    axes[2].text(0.01, 0.5, f"Beam (k={beam_size}): ${beam_pred}$", fontsize=16, color="darkgreen")
    axes[2].set_title("Beam Search Output", fontsize=10, loc='left', color='gray')

    plt.tight_layout()
    plt.show()

    print("=" * 80)
    print("Image File  :", image_path)
    if ground_truth:
        print("Ground Truth:", ground_truth)
    print("Greedy Pred :", greedy_pred)
    print(f"Beam (k={beam_size}) :", beam_pred)
    print("=" * 80)

In [ ]:
# ==========================================================
# Cell 11 - Test Prediction on a Sample Test Image
# ==========================================================

# Path to a sample test image
TEST_IMAGE_PATH = "dataset/HME100K/test/test_images/test_3.jpg"

if os.path.exists(TEST_IMAGE_PATH):
    visualize_prediction(TEST_IMAGE_PATH, beam_size=5)
else:
    # Fallback to search any image in test folder
    test_dir = Path("dataset/HME100K/test/test_images")
    if test_dir.exists():
        sample_imgs = list(test_dir.glob("*.jpg")) + list(test_dir.glob("*.png"))
        if sample_imgs:
            visualize_prediction(str(sample_imgs[0]), beam_size=5)
        else:
            print("No sample test images found in dataset/HME100K/test/test_images/")

In [ ]:
# ==========================================================
# Cell 12 - Batch Inference & Visualization on Multiple Images
# ==========================================================

test_dir = Path("dataset/HME100K/test/test_images")

if test_dir.exists():
    sample_imgs = sorted(list(test_dir.glob("*.jpg")) + list(test_dir.glob("*.png")))[:4]
    for img_path in sample_imgs:
        visualize_prediction(str(img_path), beam_size=5)
else:
    print("Test image directory not found.")